# AURA V7 / V8 / V9 — Automated Forensic Audit & Benchmark Notebook

This notebook orchestrates local and Google Colab execution of the AURA runtime across hardware discovery, dynamic model discovery, multi-prompt testing, memory budget sweeps, and CUDA GPU offloading telemetry.

In [ ]:
# Step 1: Environment & Hardware Probe
import subprocess
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

print("=== AURA HARDWARE DOCTOR ===")
res = subprocess.run(["./target/release/aura", "doctor"], capture_output=True, text=True)
print(res.stdout)

print("=== AURA GPU DOCTOR ===")
gpu_res = subprocess.run(["./target/release/aura", "gpu-doctor"], capture_output=True, text=True)
print(gpu_res.stdout)

In [ ]:
# Step 2: Dynamic Model Discovery via Ollama REST API
import urllib.request

try:
    req = urllib.request.urlopen("http://localhost:11434/api/tags")
    data = json.loads(req.read().decode('utf-8'))
    models = [m.get("name") for m in data.get("models", [])]
    print(f"✅ Discovered {len(models)} Local Models: {models}")
except Exception as e:
    print(f"Ollama REST API offline: {e}")

In [ ]:
# Step 3: Run Benchmark Suite & Generate Visualizations
if os.path.exists("benchmarks/reports/forensic_suite_results.csv"):
    df = pd.read_csv("benchmarks/reports/forensic_suite_results.csv")
    print(df.head(10))
    
    # Plot TTFT vs Decode Speed across models
    plt.figure(figsize=(10, 5))
    plt.title("AURA Benchmark Performance by Model")
    plt.xlabel("Model")
    plt.ylabel("Decode Speed (tok/s)")
    df.groupby("model")["elapsed_sec"].mean().plot(kind="bar")
    plt.grid(True)
    plt.show()
else:
    print("Run 'python benchmarks/runners/run_local.py' first to generate results.")